# Step 0 — 内存、磁盘与实验配置

对应 `reference/plan.md` 的 Step 0。本 Notebook 不删除任何缓存或模型，先回答：

- 系统内存、GPU 显存、磁盘是否足以开始；
- 是否有遗留进程占用 GPU；
- 建立后续所有 Notebook 共用的 3090/L4 单卡配置。

**通过标准：** CUDA 可用、GPU 显存约 24GB、磁盘至少预留 15GB、系统可用内存至少 12GB。

In [ ]:
from pathlib import Path
import gc
import importlib.util
import json
import shutil
import subprocess
import sys

required = ["torch", "vllm", "transformers", "datasets", "pandas", "pyarrow", "matplotlib", "sklearn", "psutil"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(f"缺少依赖 {missing}；请先执行 conda env create -f environment.yml")

import matplotlib.pyplot as plt
import pandas as pd
import psutil
import torch
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/ZIP-RC")]
REPO = next((path for path in candidates if (path / "src" / "generate_ziprc_rollouts.py").exists()), None)
if REPO is None:
    raise FileNotFoundError("找不到 ZIP-RC 仓库根目录。")
sys.path.insert(0, str(REPO / "notebooks"))
from ziprc_notebook_utils import gate, gate_frame, save_stage_report

In [ ]:
# 安全清理当前 Notebook 自己不再引用的 Python/CUDA 缓存；不会删除文件，也不会终止其他进程。
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

vm = psutil.virtual_memory()
disk = shutil.disk_usage(REPO)
cuda_ok = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_ok else "No CUDA GPU"
gpu_total_gb = torch.cuda.get_device_properties(0).total_memory / 2**30 if cuda_ok else 0.0
gpu_free_gb = torch.cuda.mem_get_info(0)[0] / 2**30 if cuda_ok else 0.0

resources = pd.DataFrame(
    [
        {"resource": "System RAM", "used_gb": (vm.total - vm.available) / 2**30, "free_gb": vm.available / 2**30},
        {"resource": "Disk", "used_gb": (disk.total - disk.free) / 2**30, "free_gb": disk.free / 2**30},
        {"resource": "GPU VRAM", "used_gb": gpu_total_gb - gpu_free_gb, "free_gb": gpu_free_gb},
    ]
)
display(resources.round(2))

ax = resources.set_index("resource")[["used_gb", "free_gb"]].plot(
    kind="barh", stacked=True, figsize=(9, 3.5), color=["#ef767a", "#49beaa"]
)
ax.set_xlabel("GB")
ax.set_title("开始实验前的资源占用")
ax.legend(["已用", "可用"], loc="lower right")
plt.tight_layout()
plt.show()

def directory_size_gb(path):
    if not path.exists():
        return 0.0
    return sum(item.stat().st_size for item in path.rglob("*") if item.is_file()) / 2**30

storage_roots = {
    "repo/data": REPO / "data",
    "repo/models": REPO / "models",
    "HF cache": Path.home() / ".cache" / "huggingface",
    "torch cache": Path.home() / ".cache" / "torch",
    "pip cache": Path.home() / ".cache" / "pip",
}
storage = pd.Series({name: directory_size_gb(path) for name, path in storage_roots.items()}).sort_values()
display(storage.rename("GB").to_frame().round(2))
storage.plot.barh(figsize=(9, 3.5), color="#f2cf5b", title="常见数据/模型缓存占用（只读检查）")
plt.xlabel("GB")
plt.tight_layout()
plt.show()

print({"gpu": gpu_name, "vram_gb": round(gpu_total_gb, 1), "bf16": torch.cuda.is_bf16_supported() if cuda_ok else False})
try:
    apps = subprocess.run(
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv,noheader"],
        check=False, capture_output=True, text=True,
    ).stdout.strip()
    print("GPU compute processes:\n", apps or "无正在运行的 compute process")
except FileNotFoundError:
    print("nvidia-smi 不可用；跳过进程列表。")

In [ ]:
# 单张 RTX 3090 / Colab L4 的默认正式小实验。后续只需修改这里并重新运行本 cell。
config = {
    "experiment_name": "qwen3_0.6b_ziprc_small_validation",
    "model_id": "Qwen/Qwen3-0.6B",
    "grader_model_id": "Qwen/Qwen2.5-Math-1.5B-Instruct",
    "dataset": "rohinm/adaptivemath",
    "split": "train",
    "prompt_column": "problem",
    "answer_column": "answer",
    "dtype": "bfloat16",
    "distribution_token_id": 151669,
    "num_length_bins": 8,
    "reward_values": [0.0, 1/6, 2/6, 3/6, 4/6, 5/6, 1.0],
    "generation_max_model_len": 4096,
    "max_output_tokens": 2048,
    "max_num_seqs": 2,
    "temperature": 1.0,
    "min_p": 0.1,
    "pilot_prompts": 200,
    "pilot_rollouts_per_prompt": 2,
    "training_prompts": 2000,
    "training_rollouts_per_prompt": 2,
    "train_fraction": 0.8,
    "validation_fraction": 0.1,
    "test_fraction": 0.1,
    "stage1_steps": 500,
    "stage2_steps": 800,
    "num_epochs": 3,
    "batch_size": 1,
    "gradient_accumulation_steps": 8,
    "stage1_learning_rate": 1e-4,
    "stage2_learning_rate": 5e-5,
    "train_max_length": 4096,
    "grader_max_model_len": 4096,
    "gpu_memory_utilization": 0.85,
    "controller_rollouts_per_prompt": 4,
    "paths": {
        "pilot": "data/pilot_rollouts.parquet",
        "full": "data/experiment_rollouts.parquet",
        "train": "data/splits/train.parquet",
        "validation": "data/splits/validation.parquet",
        "test": "data/splits/test.parquet",
        "train_value": "data/splits/train_with_value.parquet",
        "intermediate_model": "models/experiment_joint_correct",
        "final_model": "models/experiment_ziprc_final",
        "stage1_metrics": "artifacts/metrics/stage1.jsonl",
        "stage2_metrics": "artifacts/metrics/stage2.jsonl",
        "predictor_positions": "artifacts/predictor_positions.parquet",
        "predictor_metrics": "artifacts/predictor_metrics.json",
        "controller_rollouts": "data/controller_rollouts.parquet",
        "controller_scored": "data/controller_scored.parquet",
    },
}

for directory in [REPO / "data" / "splits", REPO / "models", REPO / "artifacts" / "metrics", REPO / "artifacts" / "stage_reports"]:
    directory.mkdir(parents=True, exist_ok=True)
config_path = REPO / "artifacts" / "experiment_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
display(pd.DataFrame([config]).T.rename(columns={0: "value"}))
print("Saved:", config_path)

In [ ]:
checks = [
    gate("CUDA 可用", cuda_ok, gpu_name),
    gate("24GB 级 GPU", gpu_total_gb >= 20, f"检测到 {gpu_total_gb:.1f} GB；目标为 RTX 3090/L4"),
    gate("系统可用内存", vm.available / 2**30 >= 12, f"可用 {vm.available / 2**30:.1f} GB"),
    gate("磁盘空间", disk.free / 2**30 >= 15, f"可用 {disk.free / 2**30:.1f} GB"),
    gate("BF16 可用", bool(cuda_ok and torch.cuda.is_bf16_supported()), "配置固定使用 BF16"),
    gate("配置已保存", config_path.exists(), str(config_path)),
]
display(gate_frame(checks))
report = save_stage_report(
    REPO,
    "00_memory_and_config",
    checks,
    {"gpu": gpu_name, "gpu_total_gb": gpu_total_gb, "ram_available_gb": vm.available / 2**30, "disk_free_gb": disk.free / 2**30},
)
print("Stage report:", report)